<a href="https://colab.research.google.com/github/alejitovm97-byte/Alejo-Varelas-projects/blob/gh-pages/05_factores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FACTORES v2 — panel point-in-time
# ============================================================
# Reemplaza al pipeline de 01_qmj_analysis_v3.py. El 01 queda como referencia.
#
# Que cambia y por que (detalle completo en desviaciones.md):
#
#   D-08  Universo POINT-IN-TIME reconstruido desde eventos de entrada/salida del
#         indice: 1245 RICs contra los 746 del universo viejo. Se eliminan los
#         filtros full-sample ("market cap >= 2B ever", ">= 5 anios fiscales"),
#         que eran look-ahead y borraban el 54% de los leavers & joiners.
#   D-05  Alineacion por FECHA DE ANUNCIO REAL, no por lag supuesto de 3 meses.
#   D-06  Deteccion de reexpresiones via SourceDate - anuncio.
#   D-03  Frecuencia TRIMESTRAL (Period='FQ0'), no anual.
#   D-08  Sectores TRBC en vez de GICS: GICS no clasifica instrumentos de baja y
#         usarlo habria deshecho la correccion de survivorship.
#
# CORRECCIONES DE DATOS (son errores objetivos, NO especificaciones alternativas:
# no cuentan como trials para el Deflated Sharpe):
#   D-17  Market cap VIVO en vez del rancio del ultimo anuncio. El rancio daba
#         vuelta el factor Value entero (-6.70%/anio t=-2.99 -> +2.68%).
#   D-18a Fecha de disponibilidad reparada: el 5.3% de los anuncios estaba fechado
#         ANTES del cierre del periodo, lo que inyectaba look-ahead.
#   D-21  EVOL sobre ROE trimestral, no sobre 60 filas del panel mensual.
#
# ELECCIONES DE ESPECIFICACION (estas SI cuentan como trials):
#   D-18  Z-score global del indice, no sectorial (como QMJ y como el 01).
#   D-19  Se agregan Ohlson O-Score a Safety y d_gmar a Growth, que faltaban
#         respecto de QMJ eq.3 y eq.4.
#   D-20  Safety ponderado por dimension de riesgo: lev/ohlson/altman colapsan en
#         un sub-score de solvencia para no contar tres veces la misma senal.
#   D-22  Growth standalone = sales_3y + eps_p, sin `rar` (que era momentum).
#
# EL ARCHIVO ES REPRODUCIBLE PERO NO HACE FALTA RE-CORRERLO: todo queda cacheado en
# parquet. Una sesion de analisis pone MODO='analisis' y lee de disco sin tocar
# Refinitiv.


# ============================================================
# BLOQUE 0 — Setup
# ============================================================
import re, sys, time, hashlib, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

from google.colab import drive, userdata
drive.mount('/content/drive')

ROOT_V1 = Path('/content/drive/MyDrive/TFM_regime_allocation')
ROOT    = ROOT_V1 / 'v2'
DIRS = {k: ROOT / v for k, v in {
    'raw': 'data/raw', 'processed': 'data/processed', 'config': 'config',
    'results': 'results', 'figures': 'results/figures', 'logs': 'logs'}.items()}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

MODO = 'analisis'          # 'analisis' | 'descarga'
CACHE = DIRS['raw'] / 'fund_q'
CACHE.mkdir(parents=True, exist_ok=True)

# --- conexion PEREZOSA: solo se abre si hay que descargar ---
# NOTA DE ENTORNO (costo medio dia encontrarla):
#   refinitiv-data debe instalarse con --no-deps y httpx FIJADO en 0.27.2.
#   httpx 0.28 cambio la API del proxy y rompe ses.open() con
#   AttributeError: 'dict' object has no attribute 'url'.
#   Instalar con deps hace que pip entre en backtracking y quede colgado horas.
#     !pip install -q --no-deps refinitiv-data "httpx==0.27.2" "httpcore==1.0.5" \
#         pyee watchdog appdirs tenacity nest-asyncio validators pyhumps \
#         simplejson python-configuration h11 anyio sniffio certifi idna \
#         python-dateutil pymysql
#   Tras instalar hay que REINICIAR EL KERNEL (no la VM: los paquetes viven en el
#   disco de la VM y sobreviven el reinicio del kernel).
_rd = None
def rdp():
    global _rd
    if _rd is None:
        import refinitiv.data as rd
        from refinitiv.data import session
        try:
            rd.close_session()
        except Exception:
            pass
        S = {k: userdata.get('REFINITIV_' + k)
             for k in ['APP_KEY', 'USERNAME', 'PASSWORD']}
        g = session.platform.GrantPassword(username=S['USERNAME'],
                                           password=S['PASSWORD'])
        ses = session.platform.Definition(app_key=S['APP_KEY'],
                                          grant=g).get_session()
        session.set_default(ses)
        ses.open()
        print('RDP:', ses.open_state)
        _rd = rd
    return _rd

Mounted at /content/drive


In [ ]:
# ============================================================
# BLOQUE 1 — Universo point-in-time
# ============================================================
# La consulta directa de constituyentes historicos (TR.IndexConstituentRIC con
# SDate pasado) devuelve VACIO. Los eventos de entrada/salida si funcionan, con
# tres trucos que costo encontrar:
#  1. TR.IndexJLFlag NO devuelve nada, asi que no hay campo que distinga entrada de
#     salida. Se corre la consulta DOS VECES, con IC='J' y IC='L': las etiquetas
#     salen por construccion.
#  2. Hay que pedirlo en ventanas de 5 anios; 35 de una vez da ReadTimeout.
#  3. El primer dia (1994-12-30) concentra 483 eventos, todos de entrada: no son
#     eventos, es la FOTO INICIAL de miembros. El snapshot que la consulta directa
#     negaba viene igual, disfrazado de eventos.

P_EVENTOS = DIRS['raw'] / 'f2_eventos_indice.parquet'

def bajar_eventos(ic, etiqueta):
    rd = rdp()
    partes = []
    for a in range(1994, 2027, 5):
        b = min(a + 4, 2026)
        p = rd.get_data(
            universe=['.SPX'],
            fields=['TR.IndexJLConstituentChangeDate',
                    'TR.IndexJLConstituentRIC', 'TR.IndexJLConstituentName'],
            parameters={'SDate': f'{a}-01-01', 'EDate': f'{b}-12-31', 'IC': ic})
        partes.append(p)
    d = pd.concat(partes, ignore_index=True).drop_duplicates()
    d.columns = ['indice', 'fecha', 'ric', 'nombre'][:len(d.columns)]
    d['fecha'] = pd.to_datetime(d['fecha'], errors='coerce')
    d = d.dropna(subset=['fecha', 'ric'])
    d['tipo'] = etiqueta
    return d

if MODO == 'descarga' or not P_EVENTOS.exists():
    jl = pd.concat([bajar_eventos('J', 'entrada'),
                    bajar_eventos('L', 'salida')], ignore_index=True)
    jl.sort_values('fecha').to_parquet(P_EVENTOS)
jl = pd.read_parquet(P_EVENTOS)
print(f'eventos: {len(jl)} | RICs: {jl.ric.nunique()}')

# --- reconstruccion de la membresia ---
miembros, historia, sin_entrada = set(), [], set()
for fecha, grupo in jl.sort_values(['fecha', 'tipo']).groupby('fecha'):
    # las salidas van antes que las entradas del mismo dia, por si un RIC sale y
    # vuelve a entrar en la misma fecha
    for _, r in grupo[grupo.tipo == 'salida'].iterrows():
        if r.ric not in miembros:
            sin_entrada.add(r.ric)       # era miembro desde antes del snapshot
        miembros.discard(r.ric)
    for _, r in grupo[grupo.tipo == 'entrada'].iterrows():
        miembros.add(r.ric)
    historia.append({'fecha': fecha, 'miembros': frozenset(miembros)})
H = pd.DataFrame(historia).set_index('fecha')

filas = []
for m in pd.date_range('1995-01-31', '2026-07-31', freq='ME'):
    prev = H.index[H.index <= m]
    if len(prev):
        filas.extend((m, r) for r in H.loc[prev.max(), 'miembros'])
univ = pd.DataFrame(filas, columns=['mes', 'ric'])

# --- correccion: ningun RIC puede ser miembro despues de su baja ---
# El sufijo `^` codifica el MES DE BAJA (A=ene ... L=dic + anio). Mapeo verificado
# en D-09 contra cuatro casos reales (BBBY may-2023, ATVI oct-2023, CELG nov-2019,
# MON jun-2018). Sin esto, EVHC.N^L16 y MHS_w.N^H03 quedaban pegados hasta 2026
# porque tenian entrada y nunca salida.
MES = dict(zip('ABCDEFGHIJKL', range(1, 13)))
def fecha_baja(ric):
    m = re.search(r'\^([A-L])(\d{2})$', ric)
    if not m:
        return pd.NaT
    mm, yy = MES[m.group(1)], int(m.group(2))
    return pd.Timestamp(1900 + yy if yy > 30 else 2000 + yy, mm, 1)

baja = univ.ric.map(fecha_baja)
univ = univ[~(baja.notna() & (univ.mes > baja))]
univ.to_parquet(DIRS['processed'] / 'f2_universo_pit.parquet')

# --- AUDITORIA: el S&P 500 tuvo 500 miembros toda la muestra ---
n_mes = univ.groupby('mes').size()
anual = n_mes.groupby(n_mes.index.year).last()
fuera = anual[(anual < 480) | (anual > 520)]
print(f'universo: {len(univ):,} pares | {univ.ric.nunique()} RICs')
print(f'anios fuera de 480-520: {len(fuera)}  {"OK" if not len(fuera) else "REVISAR"}')
print(f'RICs que salen sin entrada previa: {len(sin_entrada)}')
# El conteo sube a 505-509 desde 2015 y NO es un error: el indice tiene 500
# EMPRESAS pero mas CLASES DE ACCION (GOOG/GOOGL, FOX/FOXA, NWS/NWSA), y aparece
# exactamente cuando esas clases duales entraron.

eventos: 2163 | RICs: 1287
universo: 189,204 pares | 1242 RICs
anios fuera de 480-520: 0  OK
RICs que salen sin entrada previa: 3


In [ ]:
# ============================================================
# BLOQUE 2 — Fundamentales trimestrales
# ============================================================
# SINTAXIS: Period='FQ0' es lo que hace que devuelva TRIMESTRES. Sin ese parametro
# devuelve anuales aunque se pida Frq='FQ'; y 'FQ' a secas en Period es rechazado.
# Control automatico: la facturacion trimestral de Apple ronda los 95 mil millones
# y la anual los 390.
#
# CAMPOS: separados en NUCLEO (cobertura universal) y SECTORIAL (huecos por tipo de
# empresa). Medido sobre 8 empresas de sectores distintos:
#   - bancos y aseguradora: sin Revenue, GrossProfit, activos corrientes, working
#     capital, cash&ST, capex
#   - utility: sin GrossProfit
#   - REIT: sin activos/pasivos corrientes ni working capital
# Por eso los factores se construyen sobre el NUCLEO y lo sectorial es un anadido:
# restringir variables es mejor que excluir el 18% del indice.
NUCLEO = [
    'TR.GICSSector', 'TR.CommonName',
    'TR.CompanyMarketCap', 'TR.PriceClose', 'TR.CompanySharesOutstanding',
    'TR.NetIncome', 'TR.TotalEquity', 'TR.TotalAssets',
    'TR.CashFromOperatingAct', 'TR.TotalDebt', 'TR.TotalLiabilities',
    'TR.RetainedEarnings', 'TR.EBIT',
    'TR.F.PeriodEndDate', 'TR.F.OriginalAnnouncementDate', 'TR.F.SourceDate',
]
SECTORIAL = [
    'TR.Revenue', 'TR.GrossProfit', 'TR.TotalCurrentAssets',
    'TR.TotalCurrentLiabilities', 'TR.WorkingCapital',
    'TR.CashAndSTInvestments', 'TR.CapitalExpenditures', 'TR.InterestExpense',
]
# NO se piden y se CONSTRUYEN en su lugar:
#   TR.Beta        -> no resuelve, y D-02 lo daba por sospechoso de ser una beta
#                     ajustada tipo Blume. Se construye a la Frazzini-Pedersen
#                     desde retornos (correlacion x ratio de volatilidades).
#   TR.BasicEPS    -> no resuelve. EPS = NetIncome / CompanySharesOutstanding.
#   TR.DepreciationAmort -> vacio, y ya no hace falta: con Hribar-Collins los
#                     accruals son NI - CFO y D&A dejo de intervenir (D-01).

PAR = {'Period': 'FQ0', 'SDate': '0', 'EDate': '-150', 'Frq': 'FQ'}
RICS = sorted(univ.ric.unique())
lotes = [RICS[i:i + 30] for i in range(0, len(RICS), 30)]

def clave_lote(sub):
    """Nombre del cache por CONTENIDO, no por indice.
    Si cambia la lista de RICs los lotes se recomponen, y un cache indexado por
    posicion haria que 'lote_005' contenga empresas distintas de las que bajo.
    Corrupcion silenciosa. Con hash del contenido, un lote nuevo es un archivo
    nuevo y uno viejo solo se reusa si es exactamente el mismo conjunto."""
    return hashlib.md5('|'.join(sorted(sub)).encode()).hexdigest()[:12]

if MODO == 'descarga':
    rd = rdp()
    for i, sub in enumerate(lotes, 1):
        p = CACHE / f'lote_{clave_lote(sub)}.parquet'
        if p.exists():
            continue
        try:
            rd.get_data(sub, NUCLEO + SECTORIAL, PAR).to_parquet(p)
            print(i, 'OK')
        except Exception as e:
            print(i, 'FALLA', str(e)[:60])
        time.sleep(0.3)

P_FUND = DIRS['processed'] / 'f2_fundamentales_q.parquet'
if MODO == 'descarga' or not P_FUND.exists():
    fund = pd.concat([pd.read_parquet(p) for p in sorted(CACHE.glob('*.parquet'))],
                     ignore_index=True).drop_duplicates()
    fund['pe'] = pd.to_datetime(fund['Period End Date'], errors='coerce')
    fund['an'] = pd.to_datetime(fund['Original Announcement Date Time'],
                                errors='coerce')
    fund['sd'] = pd.to_datetime(fund['Source Filing Date Time'], errors='coerce')
    fund.to_parquet(P_FUND)
fund = pd.read_parquet(P_FUND)
print(f'fundamentales: {len(fund):,} filas | {fund.Instrument.nunique()} RICs | '
      f'{fund.pe.min().date()} -> {fund.pe.max().date()}')

# FECHA DE DISPONIBILIDAD (D-17). El 5.3% de los anuncios tiene fecha ANTERIOR al
# cierre del periodo, lo cual es imposible: nadie publica resultados antes de que
# termine el trimestre. Son casi todos RICs dados de baja (sufijo ^) y de antes de
# 2013; el peor caso esta fechado 2.987 dias antes del cierre. Como el merge_asof
# empalma por esta fecha, esos registros pegaban fundamentales a meses en los que
# todavia no eran publicos: look-ahead puro.
# SourceDate no tiene el problema (100% de cobertura, mediana 45 dias, CERO
# negativos), asi que se usa de respaldo solo donde el anuncio es imposible.
# `an` se conserva intacto para que el flag de reexpresion (D-06) siga midiendo lo
# que medía.
fund['fecha_disp'] = fund.an.where(fund.an >= fund.pe, fund.sd)
_mal = (fund.an < fund.pe).sum()
print(f'anuncios imposibles reparados con SourceDate: {_mal:,} '
      f'({_mal / len(fund) * 100:.1f}%) | lag final mediano '
      f'{(fund.fecha_disp - fund.pe).dt.days.median():.0f} dias')

# GICSSector y CommonName son campos ESTATICOS: Refinitiv los devuelve una vez por
# instrumento y no por periodo, asi que en el panel aparecen casi todos NaN. Hay que
# propagarlos por RIC y descartar las filas sin periodo.
est = fund.groupby('Instrument')[['GICS Sector Name', 'Company Common Name']].first()
f = (fund.dropna(subset=['pe'])
         .drop(columns=['GICS Sector Name', 'Company Common Name'])
         .merge(est, left_on='Instrument', right_index=True, how='left'))
print(f'con periodo: {len(f):,} filas')


fundamentales: 146,492 filas | 1245 RICs | 1988-12-31 -> 2026-08-02
anuncios imposibles reparados con SourceDate: 5,199 (3.5%) | lag final mediano 32 dias
con periodo: 119,893 filas


In [ ]:
# ============================================================
# BLOQUE 3 — Sectores (TRBC, no GICS)
# ============================================================
# GICS no clasifica instrumentos de baja: de 1206 RICs, los 641 vivos TODOS tienen
# sector y 478 de los 565 muertos NO. Descartar los sin sector habria dejado ~728
# RICs, casi exactamente los 746 del universo viejo: habria deshecho entero el
# trabajo de D-08.
# TRBC es el esquema propio de Refinitiv, no depende de la licencia de S&P, y cubre
# el 99.8%. Se usa para TODOS —no TRBC para muertos y GICS para vivos—, porque
# mezclar esquemas haria que "sector" signifique cosas distintas segun si la empresa
# sobrevivio, que es un sesgo justo en la dimension por la que normalizamos.
P_SEC = DIRS['processed'] / 'f2_sectores.parquet'
if MODO == 'descarga' or not P_SEC.exists():
    rd = rdp()
    sec = pd.concat([rd.get_data(RICS[i:i + 40],
                                 ['TR.TRBCEconomicSector', 'TR.GICSSector'])
                     for i in range(0, len(RICS), 40)], ignore_index=True)
    sec = sec.drop_duplicates('Instrument')
    sec.columns = ['ric', 'trbc', 'gics']
    # 3 empresas no alcanzan para un z-score sectorial. Se fusiona con Consumer
    # Cyclicals, que es lo consistente con GICS (clasifica las educativas como
    # Consumer Discretionary).
    sec['trbc'] = sec.trbc.replace(
        {'Academic & Educational Services': 'Consumer Cyclicals'})
    sec.to_parquet(P_SEC)
sec = pd.read_parquet(P_SEC)
print(f'sectores: TRBC {sec.trbc.notna().mean():.3f} | GICS {sec.gics.notna().mean():.3f}')


sectores: TRBC 0.998 | GICS 0.590


In [ ]:
# ============================================================
# BLOQUE 4 — Panel point-in-time
# ============================================================
# EL CORAZON DE D-05: para cada fin de mes, el ultimo fundamental CUYO ANUNCIO fue
# anterior o igual a esa fecha. Sin lag supuesto: fecha real.
#
# ARRANQUE EN 1997, por regla empirica y no por anio elegido a dedo. El escalon de
# cobertura de fecha de anuncio es nitido:
#   1995: 19.0% | 1996: 69.0% | 1997: 92.4% | 1998: 97.2% | 1999+: 98-99%
# Arrancar en 1996 meteria sesgo de seleccion: el tercio sin anuncio no es aleatorio,
# son las empresas mas chicas y nuevas, las que menos metadata historica tienen.
INICIO_PANEL = '1997-01-31'
DIAS_RANCIO  = 180     # generoso a proposito: filtrar de mas volveria a expulsar a
                       # las empresas en problemas, que es el sesgo que D-08 corrige

u2 = univ[univ.mes >= INICIO_PANEL].copy()
u2['ric'] = u2.ric.astype(str)                 # merge_asof exige dtypes iguales;
u2['mes'] = pd.to_datetime(u2.mes)             # Instrument viene como string[pyarrow]
u2 = u2.sort_values('mes').reset_index(drop=True)

f2 = f.dropna(subset=['fecha_disp']).copy()
f2['Instrument'] = f2.Instrument.astype(str)
f2['fecha_disp'] = pd.to_datetime(f2.fecha_disp)
f2 = f2.sort_values('fecha_disp').reset_index(drop=True)

# Se empalma por FECHA DE DISPONIBILIDAD, no por `an` crudo: ver D-17 en el Bloque 2.
panel = pd.merge_asof(u2, f2, left_on='mes', right_on='fecha_disp',
                      left_by='ric', right_by='Instrument',
                      direction='backward')
panel['edad'] = (panel.mes - panel.fecha_disp).dt.days
# D-06: la brecha entre SourceDate y anuncio marca si ese periodo fue REEXPRESADO
# despues. SourceDate del periodo P es la fecha del anuncio del periodo P+4
# trimestres, cuando esas cifras se republicaron como comparativo.
panel['reexpresado'] = (panel.sd - panel.an).dt.days > 30

panel_pit = panel[panel.fecha_disp.notna() & (panel.edad <= DIAS_RANCIO)]
panel_pit = panel_pit.merge(sec[['ric', 'trbc']], on='ric', how='left')
panel_pit.to_parquet(DIRS['processed'] / 'f2_panel_pit.parquet')

# --- CONTROLES ---
print(f'\npanel: {len(panel_pit):,} filas ({len(panel_pit)/len(panel)*100:.1f}% del bruto)')
print(f'con datos del futuro: {(panel_pit.fecha_disp > panel_pit.mes).sum()}  <- tiene que ser 0')
print(f'edad del dato: mediana {panel_pit.edad.median():.0f} dias, '
      f'cuartiles {panel_pit.edad.quantile(.25):.0f} y {panel_pit.edad.quantile(.75):.0f}')
print(f'reexpresados: {panel_pit.reexpresado.mean()*100:.1f}%')
print(f'sin sector: {panel_pit.trbc.isna().mean()*100:.1f}%')
print('\nmediana de edad por anio (debe ser estable, 39-44 dias):')
print(panel_pit.groupby(panel_pit.mes.dt.year)['edad'].median().tail(10).to_string())

print('\nBLOQUE 4 COMPLETO.')



panel: 174,058 filas (98.0% del bruto)
con datos del futuro: 0  <- tiene que ser 0
edad del dato: mediana 41 dias, cuartiles 21 y 66
reexpresados: 36.3%
sin sector: 0.0%

mediana de edad por anio (debe ser estable, 39-44 dias):
mes
2017    41.0
2018    42.0
2019    43.0
2020    41.0
2021    40.0
2022    42.0
2023    43.0
2024    44.0
2025    43.0
2026    41.0

BLOQUE 4 COMPLETO.


In [ ]:
# ============================================================
# BLOQUE 5 — Retornos mensuales
# ============================================================
# Necesarios para CUATRO cosas, no una:
#   - Momentum (12 meses menos el ultimo)
#   - Low Vol (volatilidad a 12 meses)
#   - Beta a la Frazzini-Pedersen, que es como se resuelve D-02 sin usar el campo
#     Beta de Refinitiv (sospechoso de ser una beta ajustada tipo Blume)
#   - Retornos FORWARD: la variable dependiente de todo el analisis
#
# SE BAJAN DOS CAMPOS A PROPOSITO:
#   TR.TotalReturn1Mo -> incluye dividendos. Es el correcto: omitirlos sesga
#                        sistematicamente contra los sectores que mas pagan
#                        (utilities, staples, REITs).
#   TR.PriceClose     -> para CONTRASTAR. En el sondeo sobre BBBY la mitad de los
#                        meses coincidian al tercer decimal y la otra mitad se
#                        apartaban diez puntos, sin dividendos de por medio que lo
#                        expliquen. No se diagnostica sobre una empresa: se baja el
#                        segundo campo y se mide el desacuerdo sobre las 1242.
#
# ARRANQUE EN 1995: el panel empieza en 1997 pero momentum necesita 12 meses de
# historia previa, y la beta a la FP mas todavia.
#
# TRUNCADO EN LA BAJA: TR.PriceClose se RELLENA HACIA ADELANTE tras el delisting.
# En BBBY las diez ultimas filas repiten 2023-05-02 con 0.0751. Sin truncar habria
# meses de retorno cero para una empresa que ya no existe, y apareceria como
# mantenible en cartera indefinidamente. TR.TotalReturn1Mo devuelve <NA> y se corta
# solo, pero se trunca igual por si acaso.
# NOTA: el problema de Shumway NO aplica aca. La serie mensual trae el desplome
# completo mes a mes (BBBY: 4.97 -> 0.0751, un -98.5%), asi que el retorno terminal
# entra solo. El 01 lo perdia por como emparejaba el dato anual, no por falta de dato.

P_RET = DIRS['processed'] / 'f2_retornos_m.parquet'
CACHE_R = DIRS['raw'] / 'ret_m'
CACHE_R.mkdir(parents=True, exist_ok=True)

CAMPOS_R = ['TR.PriceClose.date', 'TR.PriceClose', 'TR.TotalReturn1Mo']
PAR_R = {'SDate': '1995-01-01', 'EDate': '2026-08-31', 'Frq': 'M'}

if MODO == 'descarga':
    rd = rdp()
    for i, sub in enumerate(lotes, 1):
        p = CACHE_R / f'lote_{clave_lote(sub)}.parquet'
        if p.exists():
            continue
        try:
            rd.get_data(sub, CAMPOS_R, PAR_R).to_parquet(p)
            print(i, 'OK')
        except Exception as e:
            print(i, 'FALLA', str(e)[:60])
        time.sleep(0.3)

if MODO == 'descarga' or not P_RET.exists():
    archivos = sorted(CACHE_R.glob('*.parquet'))
    if not archivos:
        raise RuntimeError(
            f'No hay lotes en {CACHE_R} y tampoco existe {P_RET.name}.\n'
            f'  Poner MODO = "descarga" y correr el bloque de nuevo: con MODO en '
            f'"analisis" el bucle de descarga se saltea y no hay nada que consolidar.')
    ret = pd.concat([pd.read_parquet(p) for p in archivos], ignore_index=True)
    ret.columns = ['ric', 'fecha', 'precio', 'ret_total'][:len(ret.columns)]
    ret['fecha'] = pd.to_datetime(ret['fecha'], errors='coerce')
    ret = ret.dropna(subset=['fecha', 'ric'])
    # el relleno post-delisting genera fechas repetidas: una sola fila por mes
    ret['mes'] = ret.fecha + pd.offsets.MonthEnd(0)
    ret = ret.sort_values(['ric', 'fecha']).drop_duplicates(['ric', 'mes'], keep='first')
    # truncado en la baja usando el sufijo del RIC
    b = ret.ric.map(fecha_baja)
    ret = ret[~(b.notna() & (ret.mes > b + pd.offsets.MonthEnd(0)))]
    ret.to_parquet(P_RET)
ret = pd.read_parquet(P_RET)
print(f'retornos: {len(ret):,} filas | {ret.ric.nunique()} RICs | '
      f'{ret.mes.min().date()} -> {ret.mes.max().date()}')


# --- CONTROL DE CONSISTENCIA entre los dos campos ---
# Sin dividendos, el retorno de precio y el total deben coincidir. Donde no
# coincidan hay que saber si son casos raros o un problema estructural.
r = ret.sort_values(['ric', 'mes']).copy()
r['ret_precio'] = r.groupby('ric')['precio'].pct_change() * 100
c = r.dropna(subset=['ret_precio', 'ret_total'])
c = c[(c.ret_precio.abs() < 200) & (c.ret_total.abs() < 200)]   # saca splits mal ajustados
dif = (c.ret_total - c.ret_precio).abs()

print(f'\ncomparables: {len(c):,}')
print(f'  correlacion            : {c.ret_total.corr(c.ret_precio):.4f}')
print(f'  diferencia media abs   : {dif.mean():.3f} pp')
print(f'  |dif| > 1pp            : {(dif > 1).mean()*100:.1f}%')
print(f'  |dif| > 5pp            : {(dif > 5).mean()*100:.1f}%')
print('\ndesacuerdo >1pp por anio:')
print(c.assign(d=dif > 1).groupby(c.mes.dt.year)['d'].mean().mul(100).round(1)
       .tail(10).to_string())
# Si el desacuerdo es bajo y sin tendencia, se usa ret_total y listo.
# Si es alto o crece en algun tramo, hay que entender por que ANTES de construir
# momentum y low vol encima.

print('\nBLOQUE 5 COMPLETO.')


retornos: 290,188 filas | 1229 RICs | 1995-01-31 -> 2026-08-31

comparables: 288,937
  correlacion            : 0.9771
  diferencia media abs   : 0.754 pp
  |dif| > 1pp            : 20.1%
  |dif| > 5pp            : 2.5%

desacuerdo >1pp por anio:
mes
2017    18.7
2018    20.4
2019    19.4
2020    22.0
2021    16.2
2022    24.5
2023    28.2
2024    20.7
2025    17.5
2026    21.6

BLOQUE 5 COMPLETO.


In [ ]:
# ============================================================
# BLOQUE 6 — Acumulacion a doce meses (TTM)
# ============================================================
# EL PASO A TRIMESTRAL OBLIGA A ESTO Y EL 01 NO TENIA QUE ENFRENTARLO.
# Las variables de FLUJO (ingresos, resultado, flujo de caja, beneficio bruto) son
# un cuarto del anio. Calcular ROE con el resultado de un trimestre sobre el
# patrimonio da la cuarta parte del verdadero, y peor: en empresas estacionales el
# ranking cambia segun en que trimestre se mire.
# Las de STOCK (activos, patrimonio, deuda) se toman tal cual: ya son una foto.
#
# DOS CAMPOS VIENEN ACUMULADOS DENTRO DEL EJERCICIO, no trimestrales:
#   'Cash from Operating Activities, Cumulative'  -> el nombre es literal
#   'Capital Expenditures, Cumulative'            -> idem, pero con signo negativo
# Verificado en AAPL (cierra en septiembre): 91.443 en FQ3, 118.254 al cierre, y
# 29.935 en el trimestre de diciembre. Se reinicia.
# Sumar eso con rolling(4) habria inflado CFOA —uno de los seis componentes del
# pilar Profitability de QMJ— de forma brutal.
# NO existe version trimestral en Refinitiv: se sondearon 6 nombres alternativos y
# el unico que responde (TR.CashFromOperatingActivities) devuelve los MISMOS
# acumulados con otro nombre.
#
# LA RECONSTRUCCION NO SE HACE DETECTANDO CAIDAS DEL ACUMULADO.
# Esa idea tiene un fallo grave: una empresa con flujo de caja NEGATIVO en un
# trimestre hace bajar el acumulado sin que haya reinicio, y la regla lo leeria al
# reves. Y las empresas con flujo negativo son justo las que importan: las que estan
# en problemas, las que el survivorship bias se lleva puestas.
# Se identifica el CIERRE DE EJERCICIO de cada empresa y se diferencia dentro del
# anio fiscal.

q = f.sort_values(['Instrument', 'pe']).copy()
q['mes_pe'] = q.pe.dt.month

COL_CFO = 'Cash from Operating Activities, Cumulative'
cae = q.groupby('Instrument')[COL_CFO].diff() < 0
mes_fin = (q[cae.shift(-1).fillna(False)]
           .groupby('Instrument')['mes_pe']
           .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan))
q['mes_fin'] = q.Instrument.map(mes_fin)
q['anio_fiscal'] = np.where(q.mes_pe > q.mes_fin, q.pe.dt.year + 1, q.pe.dt.year)
# La distribucion de cierres valida el metodo: 837 en diciembre, 67/74/82 en
# marzo/junio/septiembre, y 46 en enero — el cierre tipico del retail (Walmart,
# Target). Es una distribucion de manual.

ACUMULADOS = [COL_CFO, 'Capital Expenditures, Cumulative']
for c in ACUMULADOS:
    q[c + '_q'] = q.groupby(['Instrument', 'anio_fiscal'])[c].diff().fillna(q[c])

# --- VALIDACION: identidad contable, no aproximacion ---
# Si los cuatro trimestres reconstruidos suman el acumulado del cierre, la
# diferenciacion es correcta. Se cumple o no se cumple.
# Medido: CFO 27.479 ejercicios y CapEx 26.643, error mediano CERO y 100% por
# debajo del 0.1% en ambos.
v = q.dropna(subset=['anio_fiscal'])
for c in ACUMULADOS:
    ch = v.groupby(['Instrument', 'anio_fiscal']).agg(
        s=(c + '_q', 'sum'), n=('pe', 'size'), fin=(c, 'last'))
    ch = ch[(ch.n == 4) & (ch.fin.abs() > 0)]
    err = (ch.s - ch.fin).abs() / ch.fin.abs()
    print(f'{c[:34]:36s} {len(ch)} ejercicios | err<0.1%: {(err<0.001).mean()*100:.1f}%')

# --- TTM sobre las series CORRECTAS ---
FLUJOS_Q = {
    'Net Income Incl Extra Before Distributions': 'ni',
    'Revenue': 'rev',
    'Gross Profit': 'gp',
    'EBIT': 'ebit',
    COL_CFO + '_q': 'cfo',
    'Capital Expenditures, Cumulative_q': 'capex',
}
for col, nom in FLUJOS_Q.items():
    q[nom + '_ttm'] = (q.groupby('Instrument')[col]
                        .rolling(4, min_periods=4).sum()
                        .reset_index(level=0, drop=True))
# min_periods=4 descarta los tres primeros trimestres de cada empresa. Es correcto
# —no se acumulan doce meses con nueve— y significa que una empresa recien
# incorporada tarda un anio en tener factores fundamentales. Es lo que un inversor
# real enfrentaria.

# CONTROL DE MAGNITUD (AAPL, ultimos trimestres): rev_ttm ~416-467 mil millones,
# ni_ttm ~112-129 mil, cfo_ttm ~111-147 mil. Si cfo_ttm diera 400 mil estaria
# sumando acumulados; si diera 30 mil, tomando un solo trimestre.
for nom in FLUJOS_Q.values():
    print(f'{nom:6s} cob {q[nom + "_ttm"].notna().mean():.3f}')

print('\nBLOQUE 6 COMPLETO.')


Cash from Operating Activities, Cu   27479 ejercicios | err<0.1%: 100.0%
Capital Expenditures, Cumulative     26643 ejercicios | err<0.1%: 100.0%
ni     cob 0.956
rev    cob 0.874
gp     cob 0.803
ebit   cob 0.957
cfo    cob 0.954
capex  cob 0.928

BLOQUE 6 COMPLETO.


In [ ]:
# ============================================================
# BLOQUE 7 — Ratios, market cap vivo y normalizacion
# ============================================================
# QUE CAMBIO EN ESTA VERSION Y POR QUE (detalle en desviaciones.md):
#
#   D-17 FIX  MARKET CAP VIVO. El market cap del registro trimestral es el del
#             ultimo anuncio y se repite hasta el siguiente (mediana 41 dias de
#             rancidez, p90 83). Pero bp/ey/ry son ratios CONTRA PRECIO, y el
#             precio es justamente lo que se mueve. Peor: el sesgo tiene direccion.
#             Una accion que se desplomo tras su ultimo anuncio conserva el market
#             cap viejo y alto, aparece CARA, y despues rebota; una que subio
#             aparece BARATA y despues rinde menos. Eso daba vuelta el factor Value
#             entero: Q5-Q1 de -6.70%/anio (t=-2.99) contra +2.68% al corregirlo.
#             mc_live = market cap del anuncio x (precio del mes / precio del
#             anuncio). Ambos insumos son point-in-time, sin look-ahead.
#             Contamina tambien eps_p, el termino 0.6*mc/TL de Altman y los pesos
#             con que se arma el retorno del indice para estimar beta.
#
#   D-18 SPEC Z-SCORE GLOBAL, no sectorial. QMJ rankea en el corte transversal
#             completo y el 01 hacia lo mismo (groupby('fiscal_year') en sus trece
#             llamadas). Se elimina UMBRAL_SECTOR, que apagaba variables enteras
#             para sectores enteros: ningun Financiero tenia ry/gpoa/gmar/altman,
#             ningun REIT tenia altman, ninguna Utility gpoa/gmar.
#
#   D-19 SPEC OHLSON O-SCORE y d_gmar, que faltaban. QMJ eq.4 define Safety con
#             CINCO componentes (bab, lev, o-score, z-score, evol) y eq.3 define
#             Growth con CINCO deltas, incluido d_gmar.
#
#   D-20 SPEC SAFETY POR DIMENSION DE RIESGO. lev, ohlson y altman correlacionan
#             entre 0.507 y 0.635: son una sola senal de solvencia contada tres
#             veces, y entre las tres se llevaban 3/5 del peso aplastando a bab,
#             que es el unico componente con IC apreciable. Se colapsan en un
#             sub-score y el pilar queda con tres dimensiones de igual peso:
#             riesgo de mercado (bab), de resultados (evol), de balance (solv).
#             No se descarta ningun componente del paper; se corrige la ponderacion
#             implicita, usando el mismo anidamiento de z-scores que QMJ ya aplica
#             de componente a pilar. Ademas empareja la cobertura entre sectores:
#             Financials y REITs pasan a tener las tres dimensiones.
#
#   D-21 SPEC EVOL sobre ROE TRIMESTRAL (20 trimestres, minimo 12). Antes se
#             calculaba sobre 60 filas del panel MENSUAL, donde el valor se repite
#             hasta el anuncio siguiente. Validado: ni_ttm es exactamente la suma
#             movil de 4 trimestres del neto trimestral (error mediano 0.00000).
#
#   D-22 SPEC GROWTH standalone = sales_3y + eps_p, sin `rar`. El compuesto previo
#             estaba dominado por rar = mom/vol, que correlaciona 0.93 con el
#             factor Momentum: media un momentum ajustado por riesgo con etiqueta
#             de crecimiento. El pilar growth_qmj queda intacto dentro de Quality;
#             no se usa como factor standalone porque, siendo un tercio del
#             compuesto, correlaciona 0.81 con Quality y no serviria como lente
#             independiente en el cruce factor x sector de la Fase 3.

AC, EQ, MC, DB = 'Total Assets', 'Total Equity', 'Company Market Cap', 'Total Debt'
WC, RE, TL = 'Working Capital', 'Retained Earnings (Accumulated Deficit)', 'Total Liabilities'
PC, CA, CL = 'Price Close', 'Total Current Assets', 'Total Current Liabilities'
NI_Q = 'Net Income Incl Extra Before Distributions'

# ------------------------------------------------------------
# 7.1 — Variables que necesitan la SECUENCIA TRIMESTRAL
# ------------------------------------------------------------
# Todo lo que lleva rezago se calcula sobre `q` y NO sobre el panel mensual: alli
# el valor se repite hasta el anuncio siguiente, asi que un shift(k) serian MESES
# y no trimestres. Es la misma trampa que D-03.
gq    = q.groupby('Instrument')
ac_q  = q[AC].where(q[AC] > 0)
eq_q  = q[EQ].where(q[EQ] > 0)
tl_q  = q[TL].where(q[TL] > 0)
ca_q  = q[CA].where(q[CA] > 0)
rev_q = q.rev_ttm.where(q.rev_ttm > 0)

# --- OHLSON O-SCORE (QMJ eq.4). Negado al final: mayor = mas seguro ---
# DESVIACION (D-19b): el paper usa pretax income en el termino futl. No bajamos ese
# campo, se aproxima con EBIT. Para fidelidad exacta habria que agregar
# TR.IncomeBeforeTaxes al NUCLEO y rehacer la descarga.
_ni_lag = gq['ni_ttm'].shift(4)
_intwo  = ((q.ni_ttm < 0) & (_ni_lag < 0)).astype(float)
_chin   = (q.ni_ttm - _ni_lag) / (q.ni_ttm.abs() + _ni_lag.abs()).replace(0, np.nan)
_oeneg  = (q[TL] > q[AC]).astype(float)
q['ohlson'] = -(-1.32
    - 0.407 * np.log(ac_q)
    + 6.03  * (q[TL] / ac_q)
    - 1.43  * ((ca_q - q[CL]) / ac_q)
    + 0.076 * (q[CL] / ca_q)
    - 1.72  * _oeneg
    - 2.37  * (q.ni_ttm / ac_q)
    - 1.83  * (q.ebit_ttm / tl_q)
    + 0.285 * _intwo
    - 0.521 * _chin)

# --- GROWTH DE QMJ: cinco deltas a 5 anios (eq.3) ---
# 20 trimestres = 5 anios de CALENDARIO. El paso a trimestral da frecuencia, no
# acorta la ventana economica (D-03).
for _nom, _num, _den in [('gpoa', 'gp_ttm',  ac_q), ('roe',  'ni_ttm',  eq_q),
                         ('roa',  'ni_ttm',  ac_q), ('cfoa', 'cfo_ttm', ac_q),
                         ('gmar', 'gp_ttm',  rev_q)]:
    q['d_' + _nom] = (q[_num] - gq[_num].shift(20)) / _den.groupby(q.Instrument).shift(20)
DEL = ['d_gpoa', 'd_roe', 'd_roa', 'd_cfoa', 'd_gmar']

# --- EVOL: desvio de la ROE TRIMESTRAL, 20 trimestres, minimo 12 (D-21) ---
q['roe_q'] = q[NI_Q] / eq_q
q['evol']  = -(q.groupby('Instrument')['roe_q']
                .transform(lambda s: s.rolling(20, min_periods=12).std()))

q['sales_3y'] = q.rev_ttm / gq['rev_ttm'].shift(12) - 1      # 12 trimestres = 3 anios
q['eps_chg']  = q.ni_ttm - gq['ni_ttm'].shift(4)

# ------------------------------------------------------------
# 7.2 — Panel point-in-time mensual
# ------------------------------------------------------------
q2 = q.dropna(subset=['fecha_disp']).copy()
q2['Instrument'] = q2.Instrument.astype(str)
q2['fecha_disp'] = pd.to_datetime(q2.fecha_disp)
q2 = q2.sort_values('fecha_disp').reset_index(drop=True)

panel = pd.merge_asof(u2, q2, left_on='mes', right_on='fecha_disp',
                      left_by='ric', right_by='Instrument', direction='backward')
panel['edad'] = (panel.mes - panel.fecha_disp).dt.days
pit = panel[panel.fecha_disp.notna() & (panel.edad <= DIAS_RANCIO)].copy()
pit = pit.merge(sec[['ric', 'trbc']], on='ric', how='left')
pit = pit.merge(ret[['ric', 'mes', 'precio']], on=['ric', 'mes'], how='left')
pit = pit.sort_values(['ric', 'mes'])

# ------------------------------------------------------------
# 7.3 — Market cap vivo y ratios (D-17)
# ------------------------------------------------------------
pit['mc_live'] = (pit[MC].where(pit[MC] > 0) *
                  (pit.precio / pit[PC].where(pit[PC] > 0)))
mcl = pit.mc_live.where(pit.mc_live > 0)
eq  = pit[EQ].where(pit[EQ] > 0)
ac  = pit[AC].where(pit[AC] > 0)

# where(>0) a proposito: el ROE de una empresa con patrimonio NEGATIVO no significa
# nada —da positivo cuando pierde plata— y es un error clasico que ensucia el
# ranking justo en las empresas en problemas.
pit['bp']    = eq / mcl
pit['ey']    = pit.ni_ttm / mcl
pit['ry']    = pit.rev_ttm / mcl
pit['roe']   = pit.ni_ttm / eq
pit['roa']   = pit.ni_ttm / ac
pit['cfoa']  = pit.cfo_ttm / ac
pit['gpoa']  = pit.gp_ttm / ac
pit['gmar']  = pit.gp_ttm / pit.rev_ttm.where(pit.rev_ttm > 0)
pit['acc']   = -(pit.ni_ttm - pit.cfo_ttm) / ac        # Hribar-Collins (D-01)
pit['lev']   = -(pit[DB] / ac)
pit['eps_p'] = pit.eps_chg / mcl
pit['altman'] = (1.2 * pit[WC] / ac + 1.4 * pit[RE] / ac + 3.3 * pit.ebit_ttm / ac
                 + 0.6 * mcl / pit[TL] + 1.0 * pit.rev_ttm / ac)

print(f'\npit: {len(pit):,} filas | {pit.ric.nunique()} RICs | '
      f'edad mediana {pit.edad.median():.0f} dias | '
      f'duplicados (ric,mes): {pit.duplicated(["ric","mes"]).sum()}')

print('\nBLOQUE 7 COMPLETO.')



pit: 174,058 filas | 1164 RICs | edad mediana 41 dias | duplicados (ric,mes): 0

BLOQUE 7 COMPLETO.


In [ ]:
# ============================================================
# BLOQUE 8 — Factores desde retornos
# ============================================================
r = ret.sort_values(['ric', 'mes']).copy()
r['rt'] = r.ret_total / 100

# El retorno del indice se pondera con el MARKET CAP VIVO (D-17): con el rancio,
# los pesos quedaban desfasados hasta un trimestre y contaminaban la beta.
w = pit[['mes', 'ric', 'mc_live']].rename(columns={'mc_live': 'mc'})
m = r.merge(w, on=['mes', 'ric'], how='inner').dropna(subset=['rt', 'mc'])
mkt = (m.assign(x=m.rt * m.mc).groupby('mes')
        .apply(lambda g_: g_.x.sum() / g_.mc.sum()).rename('rm'))
r = r.merge(mkt, on='mes', how='left')
g = r.groupby('ric')

# MOMENTUM: 12 meses SALTANDO EL ULTIMO. A un mes hay reversion, no continuacion
# (Asness, Moskowitz & Pedersen). Verificado contra un recalculo por pivot
# independiente: correlacion 1.0000000000, diferencia maxima 0.0.
r['mom'] = (g['rt'].transform(lambda s: np.log1p(s).rolling(11).sum())
              .groupby(r.ric).shift(1))
r['vol'] = g['rt'].transform(lambda s: s.rolling(12, min_periods=10).std())
r['lowvol'] = -r['vol']

# BETA A LA FRAZZINI-PEDERSEN (cierra D-02). NO se usa el campo Beta de Refinitiv,
# sospechoso de ser una beta ajustada tipo Blume (mediana exactamente 1.0000).
# La correlacion se estima con ventana LARGA (60m) y la volatilidad con corta (12m):
# FP argumentan que la correlacion es la parte estable.
r['vol_m'] = g['rm'].transform(lambda s: s.rolling(12, min_periods=10).std())
r['rho'] = (r.groupby('ric').apply(lambda d_: d_.rt.rolling(60, min_periods=36).corr(d_.rm))
             .reset_index(level=0, drop=True))
r['beta'] = 0.6 * (r.rho * (r.vol / r.vol_m)) + 0.4
r['bab']  = -r.beta

COLS_R = ['mom', 'lowvol', 'vol', 'beta', 'bab']
pit = pit.drop(columns=[c for c in COLS_R if c in pit.columns])
pit = pit.merge(r[['ric', 'mes'] + COLS_R], on=['ric', 'mes'], how='left')
pit['rar'] = pit.mom / pit.vol

print(f'CONTROL beta -> mediana {pit.beta.median():.3f} | '
      f'cuartiles {pit.beta.quantile(.25):.3f} / {pit.beta.quantile(.75):.3f}')
print('\nBLOQUE 8 COMPLETO.')


CONTROL beta -> mediana 0.991 | cuartiles 0.770 / 1.256

BLOQUE 8 COMPLETO.


In [ ]:
# ============================================================
# BLOQUE 9 — Los cinco factores
# ============================================================
def rank_z(s, min_n=10):
    """z-score por RANGO, no sobre el valor crudo. Es lo que hace QMJ: los ratios
    financieros tienen colas gruesas y outliers brutales —un ROE de 400% en una
    empresa con patrimonio casi nulo— y el rango los neutraliza sin winsorizar."""
    s = s.dropna()
    if len(s) < min_n:
        return pd.Series(np.nan, index=s.index)
    r_ = s.rank()
    return (r_ - r_.mean()) / r_.std(ddof=0)


def z_global(df, vars_):
    """Rank z-score en el corte transversal COMPLETO del indice, mes a mes (D-18).
    Sin agrupar por sector: si a una empresa le falta un componente, QMJ p.15 dice
    que se promedian los que quedan, no que se apague la variable para su sector."""
    for v in vars_:
        df['z_' + v] = df[v].groupby(df.mes).transform(lambda s: rank_z(s))
    return df


def combinar(df, cols, minimo):
    """Promedio de los componentes disponibles. El MINIMO es desviacion nuestra:
    QMJ no exige ninguno. Se eligen para que Financials —que no reporta beneficio
    bruto— entre igual en los tres pilares."""
    return df[cols].mean(axis=1).where(df[cols].notna().sum(axis=1) >= minimo)


pit = z_global(pit, ['bp', 'ey', 'ry', 'roe', 'roa', 'cfoa', 'gpoa', 'gmar', 'acc',
                     'bab', 'lev', 'evol', 'altman', 'ohlson',
                     'sales_3y', 'eps_p', 'rar'] + DEL)

P_PROF = ['z_gpoa', 'z_roe', 'z_roa', 'z_cfoa', 'z_gmar', 'z_acc']
P_SOLV = ['z_lev', 'z_ohlson', 'z_altman']
P_GROW = ['z_' + v for v in DEL]

pit['value']         = combinar(pit, ['z_bp', 'z_ey', 'z_ry'], 2)
pit['profitability'] = combinar(pit, P_PROF, 4)
pit['growth_qmj']    = combinar(pit, P_GROW, 3)

# SAFETY POR DIMENSION DE RIESGO (D-20): las tres medidas de solvencia colapsan en
# una antes de entrar, para que no cuenten por tres.
pit['solv_raw'] = combinar(pit, P_SOLV, 1)
pit['z_solv']   = pit.solv_raw.groupby(pit.mes).transform(lambda s: rank_z(s))
pit['safety']   = combinar(pit, ['z_bab', 'z_evol', 'z_solv'], 2)

# QMJ estandariza cada pilar (eq.2-4) y despues el compuesto (eq.5)
for _p in ['profitability', 'growth_qmj', 'safety']:
    pit['zz_' + _p] = pit[_p].groupby(pit.mes).transform(lambda s: rank_z(s))
pit['quality_raw'] = combinar(pit, ['zz_profitability', 'zz_growth_qmj', 'zz_safety'], 2)
pit['quality']     = pit.quality_raw.groupby(pit.mes).transform(lambda s: rank_z(s))

# GROWTH standalone sin `rar` (D-22): lente independiente, no un momentum disfrazado
pit['growth'] = combinar(pit, ['z_sales_3y', 'z_eps_p'], 2)

for _v in ['mom', 'lowvol']:
    pit[_v + '_f'] = pit[_v].groupby(pit.mes).transform(lambda s: rank_z(s))
pit['momentum'] = pit.mom_f
pit['lowvol']   = pit.lowvol_f

FACT = ['value', 'quality', 'momentum', 'growth', 'lowvol']
pit.to_parquet(DIRS['processed'] / 'f2_factores.parquet')

print('\ncobertura de factores:')
print(pit[FACT].notna().mean().round(3).to_string())

print('\ncomponentes efectivos por pilar (se promedia lo que hay, no se imputa):')
for _nom, _cols in [('profitability', P_PROF), ('growth_qmj', P_GROW),
                    ('safety', ['z_bab', 'z_evol', 'z_solv'])]:
    _n = pit[_cols].notna().sum(axis=1)
    print(f'  {_nom:14s} media {_n.mean():.2f} de {len(_cols)}')

print('\ncorrelacion entre factores:')
print(pit[FACT].corr().round(2).to_string())
# VALIDACION ECONOMICA — los signos que tienen que aparecer:
#   Value vs Quality   negativo   las empresas de calidad cotizan caras (tension QMJ)
#   Value vs Momentum  negativo   hallazgo central de Asness/Moskowitz/Pedersen
#   Quality vs LowVol  positivo   las dos defensivas
# Momentum queda ortogonal a todo (|corr| <= 0.21), que es lo que lo hace valioso
# como eje independiente en el cruce factor x sector de la Fase 3.

print('\nBLOQUE 9 COMPLETO. Los cinco factores estan construidos.')



cobertura de factores:
value       0.967
quality     0.978
momentum    0.988
growth      0.857
lowvol      0.989

componentes efectivos por pilar (se promedia lo que hay, no se imputa):
  profitability  media 5.58 de 6
  growth_qmj     media 4.40 de 5
  safety         media 2.83 de 3

correlacion entre factores:
          value  quality  momentum  growth  lowvol
value      1.00    -0.30     -0.21   -0.03    0.00
quality   -0.30     1.00      0.14    0.43    0.20
momentum  -0.21     0.14      1.00    0.16    0.12
growth    -0.03     0.43      0.16    1.00   -0.04
lowvol     0.00     0.20      0.12   -0.04    1.00

BLOQUE 9 COMPLETO. Los cinco factores estan construidos.
